In [1]:
import requests
import json
import sqlite3

#API endpoit
url = "https://civicdb.org/api/graphql"
headers = {
    "Content-Type": "application/json",
}


## Fetch Variants

In [2]:

query = """
query browseVariants($after: String) {
  variants(first: 300, after: $after) {
    nodes {
      id
      name
      feature {
        id
      }
    }
    pageInfo {
      endCursor
      hasNextPage
    }
    totalCount
  }
}
"""

all_variants = []
variables = {"after": None}

while True:
    response = requests.post(url, json={'query': query, 'variables': variables}, headers=headers)
    response_json = response.json()
    
    if 'data' in response_json:
        variants = response_json["data"]["variants"]["nodes"]
        all_variants.extend(variants)
        
        page_info = response_json["data"]["variants"]["pageInfo"]
        if not page_info["hasNextPage"]:
            break
        variables["after"] = page_info["endCursor"]
    else:
        print("Error in response:", response_json.get('errors'))
        break

print(f"Total profiles fetched: {len(all_variants)}")

Total profiles fetched: 4685


In [3]:
#list of variations to exclude
VARIATIONS_TO_EXCLUDE = ["activation",
    "alteration",
    "amplification",
    "deletion",
    "depletion",
    "demethylation",
    "duplication",
    "expression",
    "fusion",
    "gain-of-function",
    "inactivation",
    "loss",
    "mutation",
    "overexpression",
    "phosphorylation",
    "promoter",
    "repeat",
    "underexpression",
    "upregulation",
    "shift"]


filtered_variants = [
    variant for variant in all_variants
    if not any(exclusion in variant['name'].lower() for exclusion in VARIATIONS_TO_EXCLUDE)
]

print(f"Total filtered variants: {len(filtered_variants)}")

Total filtered variants: 3342


In [12]:
filtered_variants

[{'id': 4217, 'name': ' L89P (c.266T>A)', 'feature': {'id': 58}},
 {'id': 4214, 'name': ' T124I (c.371C>T)', 'feature': {'id': 58}},
 {'id': 4216, 'name': ' W8* (c.23G>A)', 'feature': {'id': 58}},
 {'id': 4278, 'name': ' Y112C (c.335A>G)', 'feature': {'id': 58}},
 {'id': 4232, 'name': ' c.212+1G>T', 'feature': {'id': 6}},
 {'id': 5005, 'name': '*02:01P', 'feature': {'id': 2606}},
 {'id': 5006, 'name': '*02:02P', 'feature': {'id': 2606}},
 {'id': 5007, 'name': '*02:03P', 'feature': {'id': 2606}},
 {'id': 5008, 'name': '*02:06P', 'feature': {'id': 2606}},
 {'id': 2489, 'name': '*214C (c.641_642insC)', 'feature': {'id': 58}},
 {'id': 1988, 'name': '*214C (c.642A>T)', 'feature': {'id': 58}},
 {'id': 2488, 'name': '*214G (c.640T>G)', 'feature': {'id': 58}},
 {'id': 3196, 'name': '*214Gext*14 (c.640T>G)', 'feature': {'id': 58}},
 {'id': 1986, 'name': '*214L (c.641G>T)', 'feature': {'id': 58}},
 {'id': 1987, 'name': '*214W (c.642A>G)', 'feature': {'id': 58}},
 {'id': 748, 'name': '*757L', 'fe

In [13]:
import csv

# Specify the output CSV file path
output_file = "filtered_variants_names.csv"

# Extract the "name" fields from filtered_variants
variant_names = [variant['name'] for variant in filtered_variants]

# Write the names to the CSV file
with open(output_file, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(["Name"])  # Write the header
    for name in variant_names:
        writer.writerow([name])

print(f"Filtered variant names have been saved to {output_file}")

Filtered variant names have been saved to filtered_variants_names.csv


# store Variants in SQL db

In [14]:
#store genes

# Connect to the SQLite database (or create it if it doesn't exist)
conn = sqlite3.connect('../database.db')

with open('../genomics.sql') as f:
        conn.executescript(f.read())

cursor = conn.cursor()
# Insert the genes into the table
for node in filtered_variants:
    cursor.execute('''
    INSERT OR REPLACE INTO variants (id, name, gene_id, db_source)
    VALUES (?, ?, ?, ?)
    ''', (node['id'], node['name'], node["feature"]["id"], "civic"))

# Commit the transaction and close the connection
conn.commit()
conn.close()

## Fetch Genes


In [18]:
query = """
query browseGenes($after: String) {
    genes(first: 300, after: $after) {
        nodes {
            id
            name
            description
            variants {
                nodes {
                    id
                    name
                    molecularProfiles {
                        nodes {
                            id
                            name
                            description
                            evidenceItems {
                                nodes {
                                    id
                                    name
                                    disease {
                                        id
                                        name
                                    }
                                }
                            }
                        }
                    }
                }
            }
        }
        pageInfo {
            endCursor
            hasNextPage
        }
        totalCount
    }
}
"""


all_genes = []
variables = {"after": None}

while True:
    response = requests.post(url, json={'query': query, 'variables': variables}, headers=headers)
    response_json = response.json()
    
    if 'data' in response_json:
        genes = response_json["data"]["genes"]["nodes"]
        all_genes.extend(genes)
        
        page_info = response_json["data"]["genes"]["pageInfo"]
        if not page_info["hasNextPage"]:
            break
        variables["after"] = page_info["endCursor"]
    else:
        print("Error in response:", response_json.get('errors'))
        break

print(f"Total genes fetched: {len(all_genes)}")

Total genes fetched: 710


In [6]:
all_genes

[{'id': 4244,
  'name': 'ABCB1',
  'description': '',
  'variants': {'nodes': [{'id': 404,
     'name': 'EXPRESSION',
     'molecularProfiles': {'nodes': [{'name': 'ABCB1 EXPRESSION',
        'description': None,
        'evidenceItems': {'nodes': []}}]}},
    {'id': 263,
     'name': 'I1145I',
     'molecularProfiles': {'nodes': [{'name': 'ABCB1 I1145I',
        'description': 'ABCB1 I1145I is a germline polymorphism that has improved responses in cancer. Specifically, it can indicate sensitivity to platinum-based chemotherapies for patients with NSCLC or better outcome for HER2-positive metastatic breast cancer patients.',
        'evidenceItems': {'nodes': [{'id': 675,
           'name': 'EID675',
           'disease': {'id': 8, 'name': 'Lung Non-small Cell Carcinoma'}},
          {'id': 1076,
           'name': 'EID1076',
           'disease': {'id': 22, 'name': 'Breast Cancer'}}]}}]}},
    {'id': 2915,
     'name': 'Overexpression',
     'molecularProfiles': {'nodes': [{'name': 'A

In [7]:
#checking the disease content (mostly empty)
i = 0
for gene in all_genes:
    for variant in gene["variants"]["nodes"]:
        for profile in variant["molecularProfiles"]["nodes"]:
            if profile["evidenceItems"]["nodes"] and profile["evidenceItems"]["nodes"][0]["disease"]:
                    i = i+1
                    print(f"Gene: {gene['name']}, Profile: {profile['name']}, Disease: {profile['evidenceItems']['nodes'][0]['disease']}")
print(i)

Gene: ABCB1, Profile: ABCB1 I1145I, Disease: {'id': 8, 'name': 'Lung Non-small Cell Carcinoma'}
Gene: ABCB1, Profile: ABCB1 Overexpression, Disease: {'id': 8, 'name': 'Lung Non-small Cell Carcinoma'}
Gene: ABCB1, Profile: ABCB1 S893A/T, Disease: {'id': 22, 'name': 'Breast Cancer'}
Gene: ABCB1, Profile: ABCB1 S893T, Disease: {'id': 20, 'name': 'Ovarian Cancer'}
Gene: ABCC10, Profile: ABCC10 Overexpression, Disease: {'id': 8, 'name': 'Lung Non-small Cell Carcinoma'}
Gene: ABCC3, Profile: ABCC3 Amplification, Disease: {'id': 22, 'name': 'Breast Cancer'}
Gene: ABCG2, Profile: ABCG2 Q141K, Disease: {'id': 20, 'name': 'Ovarian Cancer'}
Gene: ABL1, Profile: BCR::ABL1 Fusion AND ABL1 A365V, Disease: {'id': 4, 'name': 'Chronic Myeloid Leukemia'}
Gene: ABL1, Profile: BCR::ABL1 Fusion AND ABL1 A366G, Disease: {'id': 4, 'name': 'Chronic Myeloid Leukemia'}
Gene: ABL1, Profile: BCR::ABL1 Fusion AND ABL1 A397P, Disease: {'id': 4, 'name': 'Chronic Myeloid Leukemia'}
Gene: ABL1, Profile: ABL1 C475V, Di

##### store genes in SQL db

In [25]:
#store genes

# Connect to the SQLite database (or create it if it doesn't exist)
conn = sqlite3.connect('../database.db')

with open('../genomics.sql') as f:
        conn.executescript(f.read())

cursor = conn.cursor()
# Insert the genes into the table
for node in all_genes:
    disease_ids = []
    variant_ids = []
    mp_ids      = []
    for variant in node["variants"]["nodes"]:
        variant_ids.append(variant["id"])
        for profile in variant["molecularProfiles"]["nodes"]:
            mp_ids.append(profile["id"])
            for evidenceItem in profile["evidenceItems"]["nodes"]:
                if evidenceItem["disease"] and evidenceItem["disease"]["id"]:
                    disease_ids.append(evidenceItem["disease"]["id"])
    disease_ids = json.dumps(list(set(disease_ids)))  # Remove duplicates and convert to JSON format
    variant_ids = json.dumps(list(set(variant_ids)))  # Remove duplicates and convert to JSON format
    mp_ids = json.dumps(list(set(mp_ids)))  # Remove duplicates and convert to JSON format

    cursor.execute('''
    INSERT OR REPLACE INTO genes (id, name, description, molecular_profiles, variants, diseases, db_source)
    VALUES (?, ?, ?, ?, ?, ?, ?)
    ''', (node['id'], node['name'], node['description'], mp_ids, variant_ids, disease_ids, "civic"))

# Commit the transaction and close the connection
conn.commit()
conn.close()

# Fetch diseases

In [9]:
query = """
query browseDiseases($after: String) {
  diseases(first: 300, after: $after) {
    nodes {
        id
        name
    }  
    pageInfo {
      endCursor
      hasNextPage
    }
    totalCount
  }
}
"""

all_diseases = []
variables = {"after": None}

while True:
    response = requests.post(url, json={'query': query, 'variables': variables}, headers=headers)
    response_json = response.json()
    
    if 'data' in response_json:
        diseases = response_json["data"]["diseases"]["nodes"]
        all_diseases.extend(diseases)
        
        page_info = response_json["data"]["diseases"]["pageInfo"]
        if not page_info["hasNextPage"]:
            break
        variables["after"] = page_info["endCursor"]
    else:
        print("Error in response:", response_json.get('errors'))
        break

print(f"Total profiles fetched: {len(all_diseases)}")

Total profiles fetched: 831


#### store diseases in SQL db

In [10]:
# Connect to the SQLite database (or create it if it doesn't exist)
conn = sqlite3.connect('../database.db')

with open('../genomics.sql') as f:
        conn.executescript(f.read())

cursor = conn.cursor()
# Insert the filtered molecular profiles into the table
for disease in all_diseases:
    cursor.execute('''
    INSERT OR REPLACE INTO diseases (id, name, db_source)
    VALUES (?, ?, ?)
    ''', (disease['id'], disease['name'], "civic"))

# Commit the transaction and close the connection
conn.commit()
conn.close()

# Fetch Molecular Profiles

In [11]:
query = """
query browseMolecularProfiles($after: String) {
  molecularProfiles(first: 300, after: $after) {
    edges {
      node {
        id
        name
        description
        molecularProfileScore
        variants {
          id
          name
          feature {
            id
            name
          }
        }
        assertions {
          nodes{
            id
            name
            description
            disease{
              id
              name
            } 
          }
        } 
      }
    }
    pageInfo {
      endCursor
      hasNextPage
    }
    totalCount
  }
}
"""

all_molecular_profiles = []
variables = {"after": None}

while True:
    response = requests.post(url, json={'query': query, 'variables': variables}, headers=headers)
    response_json = response.json()
    
    if 'data' in response_json:
        molecular_profiles = response_json["data"]["molecularProfiles"]["edges"]
        all_molecular_profiles.extend(molecular_profiles)
        
        page_info = response_json["data"]["molecularProfiles"]["pageInfo"]
        if not page_info["hasNextPage"]:
            break
        variables["after"] = page_info["endCursor"]
    else:
        print("Error in response:", response_json.get('errors'))
        break

print(f"Total profiles fetched: {len(all_molecular_profiles)}")

Total profiles fetched: 5072


##### filter MP

In [29]:
#filter out MP with score 0
molecular_profiles_filtered = [edge for edge in all_molecular_profiles if edge["node"]["molecularProfileScore"] != 0]

molecular_profiles_filtered = [
    mp for mp in molecular_profiles_filtered
    if not any(exclusion in mp["node"]['name'].lower() for exclusion in VARIATIONS_TO_EXCLUDE)
]

print(f"Total filtered profiles: {len(molecular_profiles_filtered)}")

Total filtered profiles: 1034


In [30]:
molecular_profiles_filtered

[{'node': {'id': 12,
   'name': 'BRAF V600E',
   'description': 'BRAF V600E has been shown to be recurrent in many cancer types. It is one of the most widely studied variants in cancer. This variant is correlated with poor prognosis in certain cancer types, including colorectal cancer and papillary thyroid cancer. The targeted therapeutic dabrafenib has been shown to be effective in clinical trials with an array of BRAF mutations and cancer types. Dabrafenib has also shown to be effective when combined with the MEK inhibitor trametinib in colorectal cancer and melanoma. However, in patients with TP53, CDKN2A and KRAS mutations, dabrafenib resistance has been reported. Ipilimumab, regorafenib, vemurafenib, and a number of combination therapies have been successful in treating V600E mutations. However, cetuximab and panitumumab have been largely shown to be ineffective without supplementary treatment.',
   'molecularProfileScore': 1433.5,
   'variants': [{'id': 12,
     'name': 'V600E',


##### Store MP in sql table

In [32]:

# Connect to the SQLite database (or create it if it doesn't exist)
conn = sqlite3.connect('../database.db')

with open('../genomics.sql') as f:
        conn.executescript(f.read())

cursor = conn.cursor()
# Insert the filtered molecular profiles into the table
for profile in molecular_profiles_filtered:
    node = profile['node']
    disease_name = node['assertions']['nodes'][0]['disease']['name'] if node['assertions']['nodes'] else None
    variants_name = node["variants"][0]["name"] if node['variants'] else None
    cursor.execute('''
    INSERT OR REPLACE INTO molecular_profiles (id, name, description, variants, disease, molecularProfileScore, db_source)
    VALUES (?, ?, ?, ?, ?, ?, ?)
    ''', (node['id'], node['name'], node['description'], variants_name, disease_name,  node['molecularProfileScore'], "civic"))

# Commit the transaction and close the connection
conn.commit()
conn.close()

## Test Database

In [18]:
#test db
# Connect to the SQLite database
conn = sqlite3.connect('../database.db')
cursor = conn.cursor()

# Execute a query to retrieve all data from the molecular_profiles table
cursor.execute("SELECT * FROM genes")

# Fetch all rows from the executed query
rows = cursor.fetchall()

# Display the data
for gene in rows:
    print(gene)

# Close the connection
conn.close()

('4244', 'ABCB1', '', '[2915, 451, 262, 263, 404]', None, '[8, 4, 20, 22]', 'civic')
('16656', 'ABCC10', '', '[408]', None, '[8]', 'civic')
('16535', 'ABCC11', '', '[3841]', None, '[]', 'civic')
('6906', 'ABCC3', '', '[407]', None, '[22]', 'civic')
('7451', 'ABCG2', '', '[3852, 260]', None, '[20]', 'civic')
('4', 'ABL1', 'ABL1 is most relevant to cancer in its role in the BCR-ABL fusion protein that has become a signature of chronic myeloid leukemia (CML). Cells harboring this fusion have shown sensitivity to imatinib, greatly improving the prognostic outlook of the disease. However, additional mutations in ABL1 have been shown to confer resistance to imatinib. In these resistance cases, second-generation tyrosine kinase inhibitors such as dasatinib and nilotinib have exhibited some efficacy and are currently undergoing clinical trials for treating acquired resistance in CML.', '[1536, 4609, 4610, 3, 1028, 1538, 4611, 1029, 1537, 1527, 2, 4632, 4634, 4635, 4636, 4637, 4638, 1023, 4639,

In [ ]:
gene[3]

'['

In [12]:
#check disease content
# Connect to the SQLite database
conn = sqlite3.connect('../database.db')
cursor = conn.cursor()

# Execute a query to retrieve all data from the molecular_profiles table
cursor.execute("SELECT * FROM genes")

# Fetch all rows from the executed query
rows = cursor.fetchall()

# Display the data
i = 0
for gene in rows:
    if len(gene[5])>2:
        print(gene[5])
        i = i+1

# Close the connection
conn.close()
print(i)

[2950]
[3408, 3450, 3207]
[11, 7]
[216]
[3502]
[361]
[224]
[3432, 3466, 3431]
[368]
[51]
[216, 157]
[3]
[3225]
[2953]
[2150]
[3069]
[216]
[8]
[3224]
[3241, 3387]
[3387]
[8]
[3433]
[15]
[3465, 3516, 118, 3447]
[3]
26


In [13]:
gene[5]

'[]'

In [14]:
#checking the assertions content (mostly empty)
i = 0
for gene in all_genes:
    for variant in gene["variants"]["nodes"]:
        for profile in variant["molecularProfiles"]["nodes"]:
            if profile["assertions"]["nodes"]:  # If assertions exist
                i = i+1
                print(f"Gene: {gene['name']}, Profile: {profile['name']}, Assertions: {profile['assertions']['nodes']}")
print(i)

Gene: ACVR1, Profile: ACVR1 G328V, Assertions: [{'name': 'AID9', 'description': 'ACVR1 G328V mutations occur within the kinase domain, leading to activation of downstream signaling. Exclusively seen in high-grade pediatric gliomas, supporting diagnosis of diffuse intrinsic pontine glioma.', 'disease': {'id': 2950, 'name': 'Diffuse Midline Glioma, H3 K27M-mutant'}}]
Gene: BCOR, Profile: BCOR ITD , Assertions: [{'name': 'AID95', 'description': 'CNS tumor with BCOR ITD are characterized by BCOR exon 15 ITD (internal tandem duplication). Professional guidelines (WHO) list an ITD in exon 15 of BCOR as an essential diagnostic criterion with a DNA methylation profile aligned with CNS tumor with BCOR ITD as essential for unresolved cases. BCOR ITDs are present in 3% of CNS high grade neuroepithelial tumors, with variable morphological features including a oval to spindle shaped cells, in solid growth patterns or with perivascular pseudorosettes. Gene expression of BCOR has been shown to be inc

In [15]:
#test retrieval by name
def get_all_molecular_profiles(db_path):
    # Connect to the SQLite database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Execute a query to retrieve all data from the molecular_profiles table
    cursor.execute("SELECT * FROM molecular_profiles")

    # Fetch all rows from the executed query
    rows = cursor.fetchall()

    # Close the connection
    conn.close()

    return rows

NAME = "BRAC2 Mutation"

print(rows)
    
    #if profile['node']['name'] == NAME:
    #     print(profile['node'])


[('4244', 'ABCB1', '', '[2915, 451, 262, 263, 404]', None, '[]', 'civic'), ('16656', 'ABCC10', '', '[408]', None, '[]', 'civic'), ('16535', 'ABCC11', '', '[3841]', None, '[]', 'civic'), ('6906', 'ABCC3', '', '[407]', None, '[]', 'civic'), ('7451', 'ABCG2', '', '[3852, 260]', None, '[]', 'civic'), ('4', 'ABL1', 'ABL1 is most relevant to cancer in its role in the BCR-ABL fusion protein that has become a signature of chronic myeloid leukemia (CML). Cells harboring this fusion have shown sensitivity to imatinib, greatly improving the prognostic outlook of the disease. However, additional mutations in ABL1 have been shown to confer resistance to imatinib. In these resistance cases, second-generation tyrosine kinase inhibitors such as dasatinib and nilotinib have exhibited some efficacy and are currently undergoing clinical trials for treating acquired resistance in CML.', '[1536, 4609, 4610, 3, 1028, 1538, 4611, 1029, 1537, 1527, 2, 4632, 4634, 4635, 4636, 4637, 4638, 1023, 4639, 4645, 1535

In [16]:
def get_all_molecular_profiles_with_keys(db_path):
    # Connect to the SQLite database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Execute a query to retrieve all data from the molecular_profiles table
    cursor.execute("SELECT * FROM molecular_profiles")

    # Fetch all rows from the executed query
    rows = cursor.fetchall()

    # Get the column names from the cursor description
    column_names = [description[0] for description in cursor.description]

    # Close the connection
    conn.close()

    # Combine column names with rows
    profiles_with_keys = [dict(zip(column_names, row)) for row in rows]

    return profiles_with_keys

# Use the function to get the profiles
profiles_with_keys = get_all_molecular_profiles_with_keys('../database.db')
print(profiles_with_keys)

[{'id': '12', 'name': 'BRAF V600E', 'description': 'BRAF V600E has been shown to be recurrent in many cancer types. It is one of the most widely studied variants in cancer. This variant is correlated with poor prognosis in certain cancer types, including colorectal cancer and papillary thyroid cancer. The targeted therapeutic dabrafenib has been shown to be effective in clinical trials with an array of BRAF mutations and cancer types. Dabrafenib has also shown to be effective when combined with the MEK inhibitor trametinib in colorectal cancer and melanoma. However, in patients with TP53, CDKN2A and KRAS mutations, dabrafenib resistance has been reported. Ipilimumab, regorafenib, vemurafenib, and a number of combination therapies have been successful in treating V600E mutations. However, cetuximab and panitumumab have been largely shown to be ineffective without supplementary treatment.', 'variants': 'V600E', 'TEXT': None, 'disease': 'Melanoma', 'molecularProfileScore': 1433.5, 'db_sou

In [17]:
profiles_with_keys

[{'id': '12',
  'name': 'BRAF V600E',
  'description': 'BRAF V600E has been shown to be recurrent in many cancer types. It is one of the most widely studied variants in cancer. This variant is correlated with poor prognosis in certain cancer types, including colorectal cancer and papillary thyroid cancer. The targeted therapeutic dabrafenib has been shown to be effective in clinical trials with an array of BRAF mutations and cancer types. Dabrafenib has also shown to be effective when combined with the MEK inhibitor trametinib in colorectal cancer and melanoma. However, in patients with TP53, CDKN2A and KRAS mutations, dabrafenib resistance has been reported. Ipilimumab, regorafenib, vemurafenib, and a number of combination therapies have been successful in treating V600E mutations. However, cetuximab and panitumumab have been largely shown to be ineffective without supplementary treatment.',
  'variants': 'V600E',
  'TEXT': None,
  'disease': 'Melanoma',
  'molecularProfileScore': 143

In [18]:
for profile in profiles_with_keys:
    print(profile['disease'])

Melanoma
Her2-receptor Positive Breast Cancer
Lung Non-small Cell Carcinoma
None
None
None
None
None
None
None
Acute Myeloid Leukemia
None
None
Lung Non-small Cell Carcinoma
Lung Non-small Cell Carcinoma
B-lymphoblastic Leukemia/lymphoma With BCR-ABL1
None
None
Congenital Mesoblastic Nephroma
None
None
None
None
None
Childhood B-cell Acute Lymphoblastic Leukemia
None
None
None
Acute Promyelocytic Leukemia
Lung Non-small Cell Carcinoma
None
None
None
None
None
None
None
None
None
Central Nervous System Tumor With BCOR Internal Tandem Duplication
Acute Myeloid Leukemia With CEBPA Mutation
None
None
Synovial Sarcoma
None
None
Solid Tumor
None
None
None
None
None
None
Von Hippel-Lindau Disease
Acute Myeloid Leukemia With T(8;21); (q22; Q22.1)
None
None
None
Lipofibromatosis-like Neural Tumor
Von Hippel-Lindau Disease
None
Von Hippel-Lindau Disease
Pilocytic Astrocytoma
None
Lung Non-small Cell Carcinoma
Melanoma
None
None
None
Solid Tumor
None
None
None
None
None
None
None
None
None
None
N